# Phase 6.0-B: 表观遗传标记关联分析

**项目**: Human LncRNA Atlas

**分析目标**: 揭示 lncRNA 调控位点的染色质状态特征

**数据源**: `/api/v1/export/chipseq-overlaps` API

---

## 分析流程

1. ChIP-seq 峰重叠数据获取
2. 组蛋白标记分布统计
3. 双价域（Bivalent Domain）检测
4. BA 与表观遗传标记的关系
5. 细胞类型特异性分析
6. 染色质状态与调控强度

In [ ]:
# 环境准备
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_palette("Set3")

API_BASE_URL = "http://localhost:8000/api/v1"
print("✅ 环境准备完成")

In [ ]:
# 获取多种组蛋白标记的重叠数据
marks = ['H3K4me3', 'H3K27me3', 'H3K27ac', 'H3K4me1', 'H3K36me3', 'H3K9me3']

all_overlaps = []
for mark in marks:
    response = requests.get(
        f"{API_BASE_URL}/export/chipseq-overlaps",
        params={
            "mark_names": [mark],
            "min_ba": 100,
            "limit": 10000
        }
    )
    if response.status_code == 200:
        data = response.json()['data']
        all_overlaps.extend(data)
        print(f"✅ 获取 {mark}: {len(data)} 条重叠记录")

df_chipseq = pd.DataFrame(all_overlaps)
print(f"\n总计获取 {len(df_chipseq)} 条 ChIP-seq 重叠记录")
df_chipseq.head()

## 2. 组蛋白标记分布分析

In [ ]:
# 标记类型统计
mark_counts = df_chipseq['mark_name'].value_counts()
print("组蛋白标记分布:")
print(mark_counts)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 柱状图
mark_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set3', len(mark_counts)),
                edgecolor='black')
axes[0].set_xlabel('Histone Mark', fontsize=12)
axes[0].set_ylabel('Overlap Count', fontsize=12)
axes[0].set_title('ChIP-seq Peak Overlaps by Mark Type', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# 饼图
axes[1].pie(mark_counts.values, labels=mark_counts.index, autopct='%1.1f%%',
           startangle=90, colors=sns.color_palette('Set3', len(mark_counts)))
axes[1].set_title('Mark Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/09_chipseq_mark_distribution.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存")
plt.show()

## 3. 双价域（Bivalent Domain）检测

双价域：同时具有 H3K4me3（活性标记）和 H3K27me3（抑制标记）的区域

In [ ]:
# 检测双价域
regulation_marks = df_chipseq.groupby('regulation_id')['mark_name'].apply(set).reset_index()

# 识别同时具有 H3K4me3 和 H3K27me3 的调控位点
bivalent_mask = regulation_marks['mark_name'].apply(
    lambda x: 'H3K4me3' in x and 'H3K27me3' in x
)
bivalent_regulations = regulation_marks[bivalent_mask]['regulation_id'].tolist()

print(f"双价域统计:")
print(f"  - 检测到 {len(bivalent_regulations)} 个双价域调控位点")
print(f"  - 占总调控位点的 {100 * len(bivalent_regulations) / len(regulation_marks):.2f}%")

# 获取双价域的详细信息
df_bivalent = df_chipseq[
    df_chipseq['regulation_id'].isin(bivalent_regulations)
]

# 双价域的 BA 分布
bivalent_ba = df_bivalent.drop_duplicates('regulation_id')['binding_affinity']
non_bivalent_ba = df_chipseq[
    ~df_chipseq['regulation_id'].isin(bivalent_regulations)
].drop_duplicates('regulation_id')['binding_affinity']

print(f"\n双价域 BA 统计:")
print(f"  - 平均 BA: {bivalent_ba.mean():.2f}")
print(f"  - 中位数 BA: {bivalent_ba.median():.2f}")

print(f"\n非双价域 BA 统计:")
print(f"  - 平均 BA: {non_bivalent_ba.mean():.2f}")
print(f"  - 中位数 BA: {non_bivalent_ba.median():.2f}")

# 统计检验
u_stat, p_val = stats.mannwhitneyu(bivalent_ba, non_bivalent_ba, alternative='two-sided')
print(f"\nMann-Whitney U 检验:")
print(f"  U 统计量: {u_stat:.2f}")
print(f"  p-value: {p_val:.4e}")
print(f"  结论: {'双价域与非双价域 BA 有显著差异' if p_val < 0.05 else '无显著差异'}")

In [ ]:
# 双价域可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# BA 分布对比
axes[0].hist([bivalent_ba, non_bivalent_ba], bins=30, label=['Bivalent', 'Non-Bivalent'],
            alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Binding Affinity', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('BA Distribution: Bivalent vs Non-Bivalent', fontsize=13, fontweight='bold')
axes[0].legend()

# 箱线图对比
data_for_box = pd.DataFrame({
    'BA': list(bivalent_ba) + list(non_bivalent_ba),
    'Type': ['Bivalent'] * len(bivalent_ba) + ['Non-Bivalent'] * len(non_bivalent_ba)
})
sns.boxplot(data=data_for_box, x='Type', y='BA', ax=axes[1], palette='Set2')
axes[1].set_ylabel('Binding Affinity', fontsize=12)
axes[1].set_title('BA Comparison: Bivalent vs Non-Bivalent', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/10_bivalent_domain_analysis.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存")
plt.show()

## 4. BA 与峰强度的相关性

In [ ]:
# 分析 BA 与 ChIP-seq 峰强度的关系
plt.figure(figsize=(14, 10))

for i, mark in enumerate(marks, 1):
    plt.subplot(2, 3, i)
    
    subset = df_chipseq[df_chipseq['mark_name'] == mark]
    
    if len(subset) > 0:
        plt.scatter(
            subset['binding_affinity'],
            subset['peak_score'],
            alpha=0.5,
            s=30,
            edgecolors='black',
            linewidths=0.5
        )
        
        # 计算相关系数
        corr, p_val = stats.spearmanr(subset['binding_affinity'], subset['peak_score'])
        
        plt.xlabel('Binding Affinity', fontsize=10)
        plt.ylabel('Peak Score', fontsize=10)
        plt.title(f'{mark}\nρ={corr:.3f}, p={p_val:.2e}', fontsize=11, fontweight='bold')
        plt.grid(True, alpha=0.3)

plt.suptitle('BA vs ChIP-seq Peak Score by Histone Mark', 
            fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/11_ba_vs_peak_score.png', dpi=300, bbox_inches='tight')
print("✅ 相关性分析图已保存")
plt.show()

## 5. 细胞类型分析

In [ ]:
# 细胞类型统计
cell_type_stats = df_chipseq.groupby(['cell_type', 'mark_name']).size().reset_index(name='count')
cell_type_pivot = cell_type_stats.pivot(index='cell_type', columns='mark_name', values='count').fillna(0)

print("细胞类型 × 组蛋白标记 重叠统计:")
print(cell_type_pivot.head(10))

# 热力图
plt.figure(figsize=(12, 10))
sns.heatmap(
    cell_type_pivot,
    annot=True,
    fmt='.0f',
    cmap='YlGnBu',
    linewidths=0.5,
    cbar_kws={'label': 'Overlap Count'}
)
plt.title('ChIP-seq Overlaps: Cell Type × Histone Mark', 
         fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Histone Mark', fontsize=12)
plt.ylabel('Cell Type', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('figures/12_cell_type_mark_heatmap.png', dpi=300, bbox_inches='tight')
print("✅ 热力图已保存")
plt.show()

## 6. 染色质状态分类

基于组蛋白标记组合，将调控位点分类为不同的染色质状态：
- **活性启动子**: H3K4me3 高
- **活性增强子**: H3K27ac + H3K4me1
- **抑制性**: H3K27me3 高
- **转录延伸**: H3K36me3
- **异染色质**: H3K9me3

In [ ]:
# 为每个调控位点分配染色质状态
def classify_chromatin_state(marks_set):
    """根据组蛋白标记组合分类染色质状态"""
    if 'H3K4me3' in marks_set and 'H3K27me3' in marks_set:
        return 'Bivalent'
    elif 'H3K4me3' in marks_set:
        return 'Active Promoter'
    elif 'H3K27ac' in marks_set and 'H3K4me1' in marks_set:
        return 'Active Enhancer'
    elif 'H3K27me3' in marks_set:
        return 'Repressed'
    elif 'H3K36me3' in marks_set:
        return 'Transcription'
    elif 'H3K9me3' in marks_set:
        return 'Heterochromatin'
    else:
        return 'Other'

regulation_states = df_chipseq.groupby('regulation_id')['mark_name'].apply(set).reset_index()
regulation_states['chromatin_state'] = regulation_states['mark_name'].apply(classify_chromatin_state)

state_counts = regulation_states['chromatin_state'].value_counts()
print("染色质状态分布:")
print(state_counts)

# 可视化
plt.figure(figsize=(10, 7))
state_counts.plot(kind='barh', color=sns.color_palette('coolwarm', len(state_counts)),
                 edgecolor='black')
plt.xlabel('Count', fontsize=12)
plt.ylabel('Chromatin State', fontsize=12)
plt.title('Regulatory Sites by Chromatin State', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/13_chromatin_state_classification.png', dpi=300, bbox_inches='tight')
print("✅ 染色质状态分类图已保存")
plt.show()

## 7. 关键发现总结

In [ ]:
print("=" * 80)
print("表观遗传标记关联分析 - 关键发现")
print("=" * 80)

print(f"\n1. ChIP-seq 重叠统计")
print(f"   - 总重叠记录: {len(df_chipseq)}")
print(f"   - 涉及调控位点: {df_chipseq['regulation_id'].nunique()}")
print(f"   - 主要标记: {mark_counts.index[0]} ({mark_counts.values[0]} 次重叠)")

print(f"\n2. 双价域发现")
print(f"   - 双价域数量: {len(bivalent_regulations)}")
print(f"   - 平均 BA: {bivalent_ba.mean():.2f}")
print(f"   - 统计检验: p = {p_val:.4e}")

print(f"\n3. 染色质状态分布")
for state, count in state_counts.items():
    print(f"   - {state}: {count} ({100*count/state_counts.sum():.1f}%)")

print("\n" + "=" * 80)
print("分析完成！")
print("=" * 80)